In [ ]:
import os
import requests
import pandas as pd
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings("ignore", "(Possibly )?corrupt EXIF data", UserWarning)


CSV_FILE_PATH = '/kaggle/input/datasets/hebatullahdwairi/gp2-training-data/plants_data.csv' 
OUTPUT_DIR = '/kaggle/working/ecolens_plants'
TARGET_SIZE = (224, 224)
MAX_THREADS = 16 

os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(CSV_FILE_PATH, low_memory=False)

df = df.dropna(subset=['url', 'species']) 

def download_and_process_image(row_tuple):
    """Downloads an image, resizes it, and saves it in the correct class folder."""
    index, row = row_tuple
    
    species_name = str(row['species']).strip().replace(' ', '_')
    url = row['url']
    
    species_dir = os.path.join(OUTPUT_DIR, species_name)
    os.makedirs(species_dir, exist_ok=True)
    
    save_path = os.path.join(species_dir, f"{index}.jpg")
    
    if os.path.exists(save_path):
        return True
        
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status() 
        
        img = Image.open(BytesIO(response.content)).convert('RGB')
        img = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
        img.save(save_path, 'JPEG', quality=85, optimize=True)
        return True
        
    except Exception:
        return False

print(f"Starting download process for {len(df)} images...")

rows_to_process = list(df.iterrows())

with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    results = list(tqdm(executor.map(download_and_process_image, rows_to_process), total=len(rows_to_process)))
    successful_downloads = sum(results)

print(f"Finished! Successfully downloaded and resized {successful_downloads} out of {len(rows_to_process)} images.")

Loading CSV...
Starting download process for 179959 images...


  0%|          | 0/179959 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (101756928 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (104873910 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (101082464 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (116159149 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:3452: DecompressionBombWarning: Image size (112930400 pixels) exceeds limit of 89478485 pixels, could be decompression bomb

Finished! Successfully downloaded and resized 162401 out of 179959 images.
